# 🤖 RoboQuest 2026 — ロボットを鬼ごっこで勝たせよう！

このノートブックでは **スライダーを動かすだけ** でロボットに「鬼から逃げる」動きを学習させられます。

**手順:**
1. 🔧 セットアップ実行（最初に一度だけ）
2. 🦵 歩行パラメータを設定 → 学習 → **🎮 ビューアーでリアルタイム確認**
3. 🏃 逃げパラメータを設定 → 学習 → **🎮 ビューアーでリアルタイム確認**
4. 💾 モデルを保存して提出


In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os

# ── 1. Python ライブラリのインストール ──────────────────────────────────────
print('ライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco', 'gymnasium', 'stable-baselines3[extra]',
    'mediapy', 'tqdm', 'onnx', 'onnxruntime',
    'git+https://github.com/ttktjmt/mjswan.git',
], check=True)

# ffmpeg（動画保存に必要）
subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ── 2. リポジトリのクローン（または最新化） ─────────────────────────────────
if not os.path.exists('/content/RoboQuest2026/.git'):
    print('リポジトリをダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/tadryo/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)
else:
    print('リポジトリを最新化中...')
    subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', 'origin', 'main', '-q'],
                   check=False)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# ── 3. Go2 モデルのダウンロード ─────────────────────────────────────────────
print('Go2 ロボットモデルをダウンロード中...')
subprocess.run([sys.executable, 'scripts/download_models.py'], check=True)

print('\n✅ セットアップ完了！次のセルへ進んでください。')


In [ ]:
#@title 📁 チーム名と Google Drive 接続

#@markdown チーム名を入力してください（モデルの保存先になります）
team_name = '自分のチーム名' #@param {type:"string"}

from google.colab import drive
print('Google Drive に接続します...')
drive.mount('/content/drive')

import os
SAVE_DIR = f'/content/drive/MyDrive/RoboQuest2026/{team_name}'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'\n✅ 保存先: {SAVE_DIR}')

## 🏃 フェーズ1: まず「歩く」を覚えさせよう

ロボットが鬼から逃げるためには、まず「歩く」ことを覚える必要があります。

**流れ:**
1. 歩き方の「報酬」を設定する
2. 歩行を学習させる
3. ちゃんと歩けているか確認する
4. → 次のフェーズ（鬼ごっこ）へ進む


In [ ]:
#@title 🦵 歩行パラメータの設定

#@markdown ## 速度追跡の重み
#@markdown **前後・横への速度をどれだけ正確に追う？**（大きいほど指定速度に忠実）
lin_vel_weight = 1.0 #@param {type:"slider", min:0.5, max:3.0, step:0.1}

#@markdown **回転速度をどれだけ正確に追う？**
ang_vel_weight = 0.5 #@param {type:"slider", min:0.0, max:2.0, step:0.1}

#@markdown ---
#@markdown ## 安定性の重み
#@markdown **姿勢の安定を重視する度合い**（マイナス値: 傾くほどペナルティ）
orientation_weight = -1.0 #@param {type:"slider", min:-3.0, max:0.0, step:0.1}

#@markdown **トルク（モーターの力）を節約する度合い**
torques_weight = -0.0 #@param {type:"slider", min:-0.0001, max:0.0, step:0.00001}

#@markdown **動きをなめらかにする度合い**（急な動きを減らす）
action_rate_weight = -0.05 #@param {type:"slider", min:-0.2, max:0.0, step:0.01}

#@markdown ---
#@markdown 💡 **ヒント:** まずデフォルト値で試してみよう！

print('✅ 歩行パラメータ設定完了')
print(f'  速度追跡: {lin_vel_weight}')
print(f'  回転追跡: {ang_vel_weight}')
print(f'  姿勢安定: {orientation_weight}')
print(f'  アクション滑らかさ: {action_rate_weight}')


In [ ]:
#@title ⚙️ 歩行学習の設定

#@markdown **学習ステップ数** — 大きいほど長く学習する（時間がかかる）
walk_timesteps = 300000 #@param {type:"slider", min:100000, max:1000000, step:100000}

#@markdown **並列環境数** — 同時にシミュレーションする数（多いと速い）
walk_num_envs = 4 #@param {type:"slider", min:1, max:8, step:1}

#@markdown **学習率** — 通常は変えなくてOK
walk_lr = 0.0003 #@param {type:"slider", min:0.0001, max:0.001, step:0.0001}

print(f'歩行学習設定:')
print(f'  学習ステップ数: {walk_timesteps:,}')
print(f'  並列環境数: {walk_num_envs}')
print(f'  ⏱ 目安: 約 {walk_timesteps // 100000 * 5}〜{walk_timesteps // 100000 * 15} 分')


In [ ]:
#@title 🚀 歩行学習スタート！

import os, sys
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecNormalize

from roboquest.envs.go2_walk_env import Go2WalkEnv
from roboquest.utils.reward_utils import WalkRewardConfig
from scripts.export_for_web import export_named_policy

walk_cfg = WalkRewardConfig(
    lin_vel_weight=lin_vel_weight,
    ang_vel_weight=ang_vel_weight,
    orientation_weight=orientation_weight,
    torques_weight=torques_weight,
    action_rate_weight=action_rate_weight,
)

def _make_walk_env():
    env = Go2WalkEnv(reward_config=walk_cfg, max_episode_steps=500)
    return Monitor(env, '/tmp/walk_monitor')

walk_vec_env = make_vec_env(_make_walk_env, n_envs=walk_num_envs)
walk_vec_env = VecNormalize(walk_vec_env, norm_obs=True, norm_reward=True)

walk_model = PPO(
    'MlpPolicy', walk_vec_env,
    learning_rate=walk_lr,
    n_steps=2048, batch_size=256, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, ent_coef=0.01,
    policy_kwargs={'net_arch': [256, 256, 128]},
    verbose=0,
)

print('🏃 歩行学習を開始します...')
walk_model.learn(total_timesteps=walk_timesteps, progress_bar=True)

WALK_MODEL_PATH = '/tmp/walk_model'
walk_model.save(WALK_MODEL_PATH)
walk_vec_env.save(WALK_MODEL_PATH + '_vecnorm.pkl')

print('🌐 ブラウザビューアー用 ONNX を出力します...')
export_named_policy(
    'walk',
    WALK_MODEL_PATH,
    WALK_MODEL_PATH + '_vecnorm.pkl',
    '/content/RoboQuest2026/webapp/models',
    verify=True,
)

try:
    walk_model.save(f'{SAVE_DIR}/walk_model')
except Exception:
    pass
print(f'\n✅ 歩行学習完了！ → 歩行ビューアーで確認してください')


In [ ]:
#@title 🎮 歩行ビューアー（mjswan — ブラウザ内 MuJoCo + 学習済みポリシー）
#@markdown 学習した Walk ポリシーがブラウザ内でリアルタイム動作します。
#@markdown WASD キーまたはスライダーで速度コマンドを入力してください。

import subprocess, os, sys
subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', 'origin', 'main'],
               capture_output=True)
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# キャッシュ済みモジュールをクリア
for _k in [k for k in sys.modules if k.startswith('roboquest') or k.startswith('scripts')]:
    del sys.modules[_k]

import mjswan
from scripts.build_mjswan_viewer import build_walk

app = build_walk(
    walk_onnx_path='/content/RoboQuest2026/webapp/models/walk_policy_normalized.onnx',
    output_dir='/tmp/rq_walk_dist',
)
app.launch(height=620)


## 👹 フェーズ2: 鬼ごっこで1分間逃げ切れ！

歩行を学習したロボットに、今度は「逃げ方」を覚えさせます。

**ルール:**
- 5m × 5m の壁付きアリーナ
- 1分間（3000ステップ）鬼に捕まらなければ**逃げ切り！**
- 鬼は常にロボットめがけて直進してくる
- 俯瞰カメラで2体の動きが見える


In [ ]:
#@title 🏃 逃げポリシーのパラメータ設定

#@markdown ## 逃げ方の重み
#@markdown **毎ステップの生存ボーナス** — 大きいほど「生き延びること」を重視
survival_weight = 0.5 #@param {type:"slider", min:0.0, max:2.0, step:0.1}

#@markdown **鬼との距離に比例した報酬** — 大きいほど「できるだけ離れる」を重視
distance_weight = 1.0 #@param {type:"slider", min:0.0, max:3.0, step:0.1}

#@markdown **タグされた時のペナルティ** — 大きいほど「絶対つかまりたくない」に
tag_penalty = 50.0 #@param {type:"slider", min:10.0, max:100.0, step:5.0}

#@markdown ---
#@markdown ## 鬼の設定
#@markdown **鬼の速度** — 大きいほど追いかけるのが速い（難易度UP）
oni_speed = 0.025 #@param {type:"slider", min:0.01, max:0.05, step:0.005}

#@markdown ---
#@markdown 💡 **考えてみよう:** どの設定が一番長く逃げられる？

print('✅ 逃げポリシー設定完了')
print(f'  生存ボーナス: {survival_weight}')
print(f'  距離報酬: {distance_weight}')
print(f'  タグペナルティ: {tag_penalty}')
print(f'  鬼の速度: {oni_speed} m/step')


In [ ]:
#@title ⚙️ 鬼ごっこ学習の設定

#@markdown **学習ステップ数（高レベル）** — 歩行学習より少なくても大丈夫
flee_timesteps = 200000 #@param {type:"slider", min:50000, max:500000, step:50000}

#@markdown **並列環境数** — メモリが足りない場合は 1〜2 に
flee_num_envs = 2 #@param {type:"slider", min:1, max:4, step:1}

print(f'鬼ごっこ学習設定:')
print(f'  学習ステップ数: {flee_timesteps:,}')
print(f'  並列環境数: {flee_num_envs}')
print(f'  ⏱ 目安: 約 {flee_timesteps // 50000 * 3}〜{flee_timesteps // 50000 * 8} 分')


In [ ]:
#@title 👹 鬼ごっこ学習スタート！

import os, sys
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecNormalize

from roboquest.envs.go2_tag_hierarchical_env import Go2TagHierarchicalEnv
from roboquest.utils.reward_utils import FleeRewardConfig
from scripts.export_for_web import export_all_for_web

flee_cfg = FleeRewardConfig(
    survival_weight=survival_weight,
    distance_weight=distance_weight,
    tag_penalty=tag_penalty,
)

def _make_flee_env():
    env = Go2TagHierarchicalEnv(
        low_level_model_path='/tmp/walk_model',
        flee_config=flee_cfg,
        oni_speed=oni_speed,
    )
    return Monitor(env, '/tmp/flee_monitor')

flee_vec_env = make_vec_env(_make_flee_env, n_envs=flee_num_envs)
flee_vec_env = VecNormalize(flee_vec_env, norm_obs=True, norm_reward=True)

flee_model = PPO(
    'MlpPolicy', flee_vec_env,
    learning_rate=1e-4,
    n_steps=1024, batch_size=128, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, ent_coef=0.02,
    policy_kwargs={'net_arch': [128, 128]},
    verbose=0,
)

print('👹 鬼ごっこ逃げ学習を開始します...')
flee_model.learn(total_timesteps=flee_timesteps, progress_bar=True)

FLEE_MODEL_PATH = '/tmp/flee_model'
flee_model.save(FLEE_MODEL_PATH)
flee_vec_env.save(FLEE_MODEL_PATH + '_vecnorm.pkl')

print('🌐 ブラウザビューアー用 ONNX を出力します...')
export_all_for_web(
    walk_model='/tmp/walk_model',
    walk_vecnorm='/tmp/walk_model_vecnorm.pkl',
    flee_model=FLEE_MODEL_PATH,
    flee_vecnorm=FLEE_MODEL_PATH + '_vecnorm.pkl',
    save_dir='/content/RoboQuest2026/webapp/models',
    verify=True,
)

try:
    flee_model.save(f'{SAVE_DIR}/flee_model')
    flee_vec_env.save(f'{SAVE_DIR}/flee_vecnorm.pkl')
except Exception:
    pass
print(f'\n✅ 鬼ごっこ学習完了！ → 結果を確認してください')


In [ ]:
#@title 🎮 鬼ごっこビューアー（mjswan — ブラウザ内 MuJoCo アリーナ）
#@markdown アリーナ（壁 + 鬼ボディ）を表示し、Walk ポリシーで WASD 手動操作します。
#@markdown ※ Flee AI ポリシーは今後実装予定。

import subprocess, os, sys
subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', 'origin', 'main'],
               capture_output=True)
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# キャッシュ済みモジュールをクリア
for _k in [k for k in sys.modules if k.startswith('roboquest') or k.startswith('scripts')]:
    del sys.modules[_k]

import mjswan
from scripts.build_mjswan_viewer import build_flee

app = build_flee(
    walk_onnx_path='/content/RoboQuest2026/webapp/models/walk_policy_normalized.onnx',
    output_dir='/tmp/rq_flee_dist',
)
app.launch(height=620)


In [ ]:
#@title 💾 モデルを保存して大会に提出

import os, json

params = {
    'team_name': team_name,
    'walk': {
        'lin_vel_weight': lin_vel_weight,
        'ang_vel_weight': ang_vel_weight,
        'orientation_weight': orientation_weight,
        'action_rate_weight': action_rate_weight,
        'timesteps': walk_timesteps,
    },
    'flee': {
        'survival_weight': survival_weight,
        'distance_weight': distance_weight,
        'tag_penalty': tag_penalty,
        'oni_speed': oni_speed,
        'timesteps': flee_timesteps,
    },
}

try:
    with open(f'{SAVE_DIR}/params.json', 'w', encoding='utf-8') as f:
        json.dump(params, f, ensure_ascii=False, indent=2)
    print(f'✅ パラメータを保存: {SAVE_DIR}/params.json')
    print(f'✅ 歩行モデル: {SAVE_DIR}/walk_model.zip')
    print(f'✅ 逃げモデル: {SAVE_DIR}/flee_model.zip')
    print(f'\n🎉 大会への提出準備完了！')
except Exception as e:
    print(f'Drive 保存エラー: {e}')
    print('ローカル保存済み: /tmp/walk_model.zip, /tmp/flee_model.zip')
